### Initialisation de l'environnement et Chargement des Données

Cette section prépare l'environnement de travail en installant les bibliothèques nécessaires au traitement du langage naturel (NLP) et à la manipulation des données. Elle charge ensuite le corpus parallèle Ruund-Français depuis Hugging Face et l'affiche pour une première inspection.

In [ ]:
# Installation
!pip install -q datasets pandas sentencepiece evaluate sacrebleu transformers[torch] accelerate

# Imports
from datasets import load_dataset
import pandas as pd
import evaluate
from huggingface_hub import login
from google.colab import userdata

# Token sécurisé
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

# Chargement du corpus
ds = load_dataset("eliezermga/ruund-french-parallel-corpus")

# DataFrame pour inspection
df = pd.DataFrame(ds['train'])

print(f"Taille initiale du corpus : {len(df)} paires.")
df.head()

Taille initiale du corpus : 8400 paires.


,Ruund,French
0,"Aam nid kwaam nkay chilil mwaan, chikweeza kum...",Je ne suis quant à moi qu’une simple antilope ...
1,"Aay aana, aay iin twuvum","D’un côté vous avez des enfants chéris, de l’a..."
2,"Adaap ngendj, mwiiningand madyaaj mend","Le visiteur est toujours le mieux servi, n’en ..."
3,"Adangang kwend, piikil anch afangang kwend",Mieux vaut souhaiter que le prochain vive le p...
4,"Akandiling payaaw, piikil anch ezakw kal ni mash",Mieux vaut prévenir que guérir


### Vérification de la Taille du DataFrame

Cette cellule simple permet de vérifier le nombre d'enregistrements (paires de phrases) actuellement présents dans le DataFrame après le chargement initial. C'est une étape rapide pour confirmer la taille de notre jeu de données brut.

In [ ]:
len(df)

8400

### Prétraitement du Texte pour la Normalisation

Cette étape est cruciale pour standardiser les données textuelles. La fonction `preprocess_text` applique plusieurs opérations:

1.  **Normalisation Unicode (NFC)**: Assure une représentation cohérente des caractères (e.g., les lettres accentuées).
2.  **Conversion en Minuscules**: Réduit la variation lexicale en uniformisant la casse.
3.  **Nettoyage de la Ponctuation**: Supprime les caractères spéciaux non essentiels tout en conservant la ponctuation sémantiquement utile (points, virgules, points d'interrogation, points d'exclamation, apostrophes).
4.  **Nettoyage des Espaces**: Remplace les séquences d'espaces multiples par un seul espace et supprime les espaces en début/fin de chaîne.

Ces transformations améliorent la qualité des données pour les modèles NLP en réduisant le bruit et les variations non pertinentes.

In [ ]:
import unicodedata
import re

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    # Normalisation Unicode propre
    text = unicodedata.normalize('NFC', text)

    # Lowercase
    text = text.lower()

    # Garder ponctuation utile (.,!?)
    text = re.sub(r"[^\w\s.,!?']", "", text)

    # Nettoyage espaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['Ruund'] = df['Ruund'].apply(preprocess_text)
df['French'] = df['French'].apply(preprocess_text)

### Filtrage du Corpus pour Optimisation des Données

Cette section met en œuvre une série de filtres pour affiner le corpus parallèle, visant à améliorer la qualité et la pertinence des paires de phrases pour l'entraînement d'un modèle de traduction automatique:

1.  **Suppression des Doublons**: Élimine les paires de phrases identiques pour éviter la sur-représentation et le surapprentissage.
2.  **Calcul des Longueurs**: Ajoute des colonnes `src_len` et `trg_len` indiquant le nombre de mots dans les phrases source (Ruund) et cible (Français).
3.  **Suppression des Phrases Vides**: Retire les paires où l'une des phrases est vide après le prétraitement.
4.  **Filtrage par Longueur**: Conserve uniquement les paires dont les phrases source et cible ont une longueur inférieure ou égale à 80 mots. Cela aide à gérer la complexité et les contraintes des modèles de traduction.
5.  **Filtrage par Ratio de Longueur**: Élimine les paires où le ratio de longueur entre la phrase source et cible est trop déséquilibré (e.g., une phrase très courte traduite par une phrase très longue, ou vice-versa). Un ratio de 2.0 est utilisé comme seuil pour s'assurer que les traductions restent proportionnelles.

Ces étapes de filtrage sont essentielles pour créer un jeu de données propre et équilibré, ce qui est fondamental pour l'efficacité et la robustesse des modèles NLP.

In [ ]:
# Sauvegarde taille initiale
initial_size = len(df)

# 1. Suppression doublons
df = df.drop_duplicates()

# 2. Longueurs
df['src_len'] = df['Ruund'].apply(lambda x: len(x.split()))
df['trg_len'] = df['French'].apply(lambda x: len(x.split()))

# 3. Suppression phrases vides
df = df[(df['src_len'] > 0) & (df['trg_len'] > 0)]

# 4. Filtrage longueur
df = df[(df['src_len'] <= 80) & (df['trg_len'] <= 80)]

# 5. Filtrage ratio
df = df[(df['src_len'] / df['trg_len'] <= 2.0) &
        (df['trg_len'] / df['src_len'] <= 2.0)]

# Résultat
print(f"Taille initiale : {initial_size}")
print(f"Taille après filtrage : {len(df)}")

Taille initiale : 8400
Taille après filtrage : 8087


Cette cellule prépare le modèle et le tokenizer pour la traduction automatique. Le modèle `MBartForConditionalGeneration` est un modèle multilingue pré-entraîné, idéal pour la traduction entre de nombreuses langues. Le `AutoTokenizer` associé est chargé pour convertir le texte en tokens numériques que le modèle peut comprendre. Un token spécial `ruu_CM` est ajouté pour représenter la langue Ruund, et la taille des embeddings du modèle est redimensionnée pour l'inclure. Enfin, le token de début de séquence forcé est réglé sur le français (`fr_XX`) pour s'assurer que le modèle génère des traductions dans cette langue.

In [ ]:
from transformers import MBartForConditionalGeneration, AutoTokenizer
from datasets import Dataset

model_name = "facebook/mbart-large-50"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ajout du token de langue custom pour le Ruund
tokenizer.add_special_tokens({"additional_special_tokens": ["ruu_CM"]})
tokenizer.src_lang = "ruu_CM"
tokenizer.tgt_lang = "fr_XX"

# Chargement du modèle + redimensionnement embedding pour le nouveau token
model = MBartForConditionalGeneration.from_pretrained(model_name)
model.config.tie_word_embeddings = False
model.resize_token_embeddings(len(tokenizer))

# Forcer la génération en français
model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id["fr_XX"]

print(f"Token ruu_CM id : {tokenizer.convert_tokens_to_ids('ruu_CM')}")
print(f"Taille du vocabulaire : {len(tokenizer)}")


Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~john

Token ruu_CM id : 250054
Taille du vocabulaire : 250055


Après avoir préparé le corpus, cette étape convertit le DataFrame Pandas en un objet `Dataset` de la bibliothèque `datasets`. La fonction `preprocess_function` est ensuite appliquée par lots (`batched=True`) pour tokeniser les phrases Ruund et Français. La tokenisation consiste à découper le texte en unités plus petites (mots ou sous-mots) et à les convertir en identifiants numériques. Le `padding="max_length"` assure que toutes les séquences ont la même longueur (ici, 128 tokens) en ajoutant des tokens de remplissage. Les étiquettes (phrases françaises) sont également tokenisées, et les tokens de remplissage (`padding_token_id`) sont remplacés par `-100` pour indiquer au modèle de ne pas les inclure dans le calcul de la perte, ce qui est une pratique courante pour l'entraînement des modèles séquence-à-séquence.

In [ ]:
import numpy as np
from datasets import Dataset

dataset = Dataset.from_pandas(df[['Ruund', 'French']])

def preprocess_function(examples):
    # Tokenisation source (Ruund)
    model_inputs = tokenizer(
        examples["Ruund"],
        max_length=128,
        truncation=True,
        padding=False  # ← padding délégué au DataCollator
    )

    # Tokenisation cible (Français)
    labels = tokenizer(
        text_target=examples["French"],
        max_length=128,
        truncation=True,
        padding=False  # ← idem
    )

    # Convertir en numpy array int32 directement
    model_inputs["labels"] = [
        np.array(l, dtype=np.int64) for l in labels["input_ids"]
    ]

    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["Ruund", "French"]
)

# Plus de set_format() ← le Trainer gère la conversion en tenseurs lui-même

print("Tokenisation terminée. Taille dataset :", len(tokenized_dataset))

Map:   0%|          | 0/8087 [00:00<?, ? examples/s]

Tokenisation terminée. Taille dataset : 8087


Afin d'évaluer la capacité de généralisation de notre modèle et d'éviter l'overfitting (surapprentissage), il est crucial de diviser le jeu de données en trois sous-ensembles distincts : 1. **Entraînement (`train`)**: Utilisé pour entraîner le modèle. 2. **Validation (`validation`)**: Utilisé pour ajuster les hyperparamètres et surveiller les performances du modèle pendant l'entraînement. 3. **Test (`test`)**: Utilisé pour évaluer les performances finales du modèle sur des données qu'il n'a jamais vues. Cette cellule effectue une division 80/20 pour l'entraînement/test, puis divise le jeu de test en deux pour créer des ensembles de validation et de test de taille égale. Cela garantit une évaluation robuste du modèle.

In [ ]:
from datasets import DatasetDict

train_testvalid = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
test_valid = train_testvalid["test"].train_test_split(test_size=0.5, seed=42)

final_datasets = DatasetDict({
    "train": train_testvalid["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
})

train_dataset = final_datasets["train"]
eval_dataset  = final_datasets["validation"]
test_dataset  = final_datasets["test"]

print(final_datasets)


DatasetDict({
    train: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 6469
    })
    validation: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 809
    })
    test: Dataset({
        features: ['__index_level_0__', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 809
    })
})


Cette cellule configure les outils essentiels pour l'entraînement du modèle : 1. **`DataCollatorForSeq2Seq`**: Cet objet regroupe les exemples du dataset en batches pour l'entraînement. Il est crucial pour gérer le padding dynamique, c'est-à-dire l'ajustement des séquences à la même longueur au sein de chaque batch, et pour s'assurer que les tokens de padding dans les étiquettes sont ignorés pendant le calcul de la perte (`label_pad_token_id=-100`). 2. **Métrique BLEU**: La métrique BLEU (Bilingual Evaluation Understudy) est chargée via la bibliothèque `evaluate`. C'est une métrique standard pour évaluer la qualité des traductions automatiques, qui compare la traduction générée à une ou plusieurs traductions de référence. 3. **`compute_metrics`**: Cette fonction personnalisée décode les prédictions du modèle et les labels réels, puis calcule le score BLEU. Elle est essentielle pour suivre les performances du modèle pendant l'entraînement et la validation.

In [ ]:
from transformers import DataCollatorForSeq2Seq
import evaluate
import numpy as np

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
    return_tensors="pt"
)

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Décoder les prédictions
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Remplacer -100 avant de décoder les labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # sacrebleu attend [[ref]] pour chaque prédiction
    result = metric.compute(
        predictions=decoded_preds,
        references=[[ref] for ref in decoded_labels]
    )
    return {"bleu": round(result["score"], 2)}

print("Data collator et métrique BLEU prêts.")

Data collator et métrique BLEU prêts.


Cette cellule configure et lance l'entraînement du modèle de traduction. 1. **`Seq2SeqTrainingArguments`**: Définit tous les hyperparamètres et les options d'entraînement, tels que le nombre d'époques, la taille des lots, le taux d'apprentissage, la stratégie d'évaluation et de sauvegarde, ainsi que l'optimiseur (`adafactor`). L'argument `predict_with_generate=True` est important car il garantit que la génération de texte est utilisée lors de l'évaluation, ce qui est nécessaire pour les tâches de traduction. 2. **`Seq2SeqTrainer`**: Orchestre le processus d'entraînement. Il prend le modèle, les arguments d'entraînement, les datasets (entraînement et validation), le tokenizer, le collator de données et la fonction de calcul des métriques. 3. **`trainer.train()`**: Lance le processus d'entraînement du modèle sur les données fournies. Pendant l'entraînement, le modèle apprendra à traduire du Ruund vers le Français en minimisant la fonction de perte et en essayant d'optimiser le score BLEU.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./ruund-fr-mbart",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    gradient_checkpointing=False,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=5e-5,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    bf16=True,
    fp16=False,
    optim="adafactor",
    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,2.617594,2.208037,4.670000
2,1.689691,1.728069,13.530000
3,1.257658,1.583024,17.360000
4,0.922331,1.628367,18.340000
5,0.644760,1.771419,19.730000
6,0.428646,1.903643,20.080000
7,0.291391,2.029395,20.510000
8,0.190815,2.097469,21.170000
9,0.124555,2.162725,21.740000
10,0.094730,2.200848,21.490000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight'].


TrainOutput(global_step=4050, training_loss=1.013701666843744, metrics={'train_runtime': 1953.8146, 'train_samples_per_second': 33.11, 'train_steps_per_second': 2.073, 'total_flos': 2.2355631094480896e+16, 'train_loss': 1.013701666843744, 'epoch': 10.0})

Après l'entraînement du modèle, il est crucial d'évaluer ses performances sur un jeu de données de test indépendant. Cette cellule utilise la méthode `trainer.evaluate()` pour mesurer la qualité des traductions générées par le modèle sur le `test_dataset`. Le résultat clé est le score BLEU, qui quantifie la similarité entre les traductions du modèle et les traductions de référence humaines. Un score BLEU plus élevé indique une meilleure qualité de traduction. Cette évaluation finale permet de valider la robustesse et l'efficacité du modèle sur des données qu'il n'a jamais vues, donnant une indication objective de ses capacités de généralisation.

In [ ]:
results = trainer.evaluate(test_dataset)
print(f"BLEU sur le test set : {results['eval_bleu']}")


BLEU sur le test set : 23.03


### Traduction (Inférence) avec le Modèle Entraîné

Cette cellule démontre comment utiliser le modèle de traduction entraîné pour traduire une phrase du Ruund vers le Français. Elle prend une phrase d'entrée en Ruund, la tokenise, la passe au modèle pour générer la traduction, puis décode le résultat en texte compréhensible.

In [ ]:
def translate_sentence(sentence, model, tokenizer):
    # Tokeniser la phrase source
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=128)

    # Déplacer les inputs vers le même appareil que le modèle (GPU si disponible)
    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Générer la traduction
    with torch.no_grad():
        translated_tokens = model.generate(**inputs, max_length=128, num_beams=5, early_stopping=True)

    # Décoder la traduction
    translated_sentence = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return translated_sentence

# Exemple d'utilisation
ruund_sentence = "wajidikish mwan"

# Assurez-vous que le modèle est en mode évaluation
model.eval()

# Effectuer la traduction
translated_french = translate_sentence(ruund_sentence, model, tokenizer)

print(f"Phrase Ruund : {ruund_sentence}")
print(f"Traduction Française : {translated_french}")

Phrase Ruund : wajidikish mwan
Traduction Française : le fils aîné


### Sauvegarde du Modèle Entraîné

Cette cellule permet de sauvegarder le modèle de traduction et son tokenizer associé. Il est essentiel de sauvegarder le modèle après l'entraînement pour pouvoir le réutiliser ultérieurement sans avoir à le ré-entraîner. Le tokenizer est également sauvegardé pour garantir que le modèle sera utilisé avec la même méthode de tokenisation que celle utilisée lors de son entraînement.

In [ ]:
# Chemin où sauvegarder le modèle et le tokenizer
save_path = "./fine_tuned_mbart_ruund_fr"

# Sauvegarder le modèle
model.save_pretrained(save_path)

# Sauvegarder le tokenizer
tokenizer.save_pretrained(save_path)

print(f"Modèle et tokenizer sauvegardés dans : {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modèle et tokenizer sauvegardés dans : ./fine_tuned_mbart_ruund_fr
